# Lab 3: Sales Analytics Pipeline

**Difficulty: Intermediate | ~50 min | Requires Labs 1–2**

*Lab 3 of 7 in the MongoDB Mastery series.*

In this lab, you will build a sales analytics pipeline for an e-commerce company.

You will learn how to:
1. Design a multi-collection schema (orders + products)
2. Build multi-stage aggregation pipelines to answer business questions
3. Use `$lookup` to join data across collections
4. Create indexes to speed up queries
5. Use `.explain()` to analyze query performance

In [ ]:
!pip install -qU pymongo==4.10.1 mongomock

This installs `pymongo` (the MongoDB driver) and `mongomock` (in-memory mock server) so you can practice without installing MongoDB.

### Step 1 — Connect and Create Collections

In [ ]:
import pymongo
import mongomock
from datetime import datetime

client = mongomock.MongoClient()
db = client["ecommerce"]
orders = db["orders"]
products = db["products"]

print("Connected to ecommerce database")

We create two collections: `orders` for purchase records and `products` for the product catalog. This **multi-collection** design mirrors a real e-commerce database.

### Step 2 — Seed the Product Catalog

In [ ]:
product_catalog = [
    {"product_id": "P001", "name": "Wireless Mouse",    "category": "Electronics", "price": 29.99},
    {"product_id": "P002", "name": "Mechanical Keyboard","category": "Electronics", "price": 89.99},
    {"product_id": "P003", "name": "USB-C Hub",         "category": "Electronics", "price": 45.00},
    {"product_id": "P004", "name": "Laptop Stand",      "category": "Accessories", "price": 35.00},
    {"product_id": "P005", "name": "Noise Cancelling Headphones", "category": "Electronics", "price": 199.99},
    {"product_id": "P006", "name": "Webcam HD",         "category": "Electronics", "price": 59.99},
    {"product_id": "P007", "name": "Desk Lamp",         "category": "Accessories", "price": 22.50},
    {"product_id": "P008", "name": "Monitor Arm",       "category": "Accessories", "price": 75.00},
    {"product_id": "P009", "name": "Notebook Set",      "category": "Stationery",  "price": 12.99},
    {"product_id": "P010", "name": "Pen Bundle",        "category": "Stationery",  "price": 8.50},
]

products.insert_many(product_catalog)
print(f"Inserted {products.count_documents({})} products.")

Each product has a `product_id`, `name`, `category`, and `price`. Later we will reference these `product_id` values from the orders collection.

### Step 3 — Seed Order Data

In [ ]:
order_records = [
    {"order_id": "ORD001", "customer": "Alice",   "product_id": "P001", "quantity": 2, "date": "2025-01-10", "region": "North"},
    {"order_id": "ORD002", "customer": "Bob",     "product_id": "P002", "quantity": 1, "date": "2025-01-12", "region": "South"},
    {"order_id": "ORD003", "customer": "Charlie", "product_id": "P005", "quantity": 1, "date": "2025-01-15", "region": "North"},
    {"order_id": "ORD004", "customer": "Alice",   "product_id": "P003", "quantity": 3, "date": "2025-01-20", "region": "North"},
    {"order_id": "ORD005", "customer": "Diana",   "product_id": "P004", "quantity": 1, "date": "2025-02-01", "region": "West"},
    {"order_id": "ORD006", "customer": "Eve",     "product_id": "P006", "quantity": 2, "date": "2025-02-05", "region": "East"},
    {"order_id": "ORD007", "customer": "Bob",     "product_id": "P001", "quantity": 1, "date": "2025-02-10", "region": "South"},
    {"order_id": "ORD008", "customer": "Frank",   "product_id": "P007", "quantity": 4, "date": "2025-02-14", "region": "West"},
    {"order_id": "ORD009", "customer": "Grace",   "product_id": "P002", "quantity": 1, "date": "2025-02-20", "region": "North"},
    {"order_id": "ORD010", "customer": "Alice",   "product_id": "P008", "quantity": 1, "date": "2025-03-01", "region": "North"},
    {"order_id": "ORD011", "customer": "Hank",    "product_id": "P009", "quantity": 5, "date": "2025-03-05", "region": "East"},
    {"order_id": "ORD012", "customer": "Charlie", "product_id": "P010", "quantity": 10, "date": "2025-03-10", "region": "North"},
    {"order_id": "ORD013", "customer": "Diana",   "product_id": "P005", "quantity": 1, "date": "2025-03-15", "region": "West"},
    {"order_id": "ORD014", "customer": "Eve",     "product_id": "P003", "quantity": 2, "date": "2025-03-20", "region": "East"},
    {"order_id": "ORD015", "customer": "Frank",   "product_id": "P006", "quantity": 1, "date": "2025-04-01", "region": "West"},
    {"order_id": "ORD016", "customer": "Grace",   "product_id": "P001", "quantity": 3, "date": "2025-04-05", "region": "North"},
    {"order_id": "ORD017", "customer": "Hank",    "product_id": "P004", "quantity": 2, "date": "2025-04-10", "region": "East"},
    {"order_id": "ORD018", "customer": "Alice",   "product_id": "P002", "quantity": 1, "date": "2025-04-15", "region": "North"},
    {"order_id": "ORD019", "customer": "Bob",     "product_id": "P007", "quantity": 2, "date": "2025-04-20", "region": "South"},
    {"order_id": "ORD020", "customer": "Charlie", "product_id": "P008", "quantity": 1, "date": "2025-05-01", "region": "North"},
]

orders.insert_many(order_records)
print(f"Inserted {orders.count_documents({})} orders.")

Each order links to a product via `product_id`. Orders span January–May 2025 across four regions. We will use aggregation pipelines to slice this data in different ways.

### Step 4 — Revenue by Product (Multi-Stage Pipeline)

In [ ]:
# Pipeline: join with products, compute revenue per order, then sum by product
pipeline = [
    {"$lookup": {
        "from": "products",
        "localField": "product_id",
        "foreignField": "product_id",
        "as": "product"
    }},
    {"$unwind": "$product"},
    {"$addFields": {
        "revenue": {"$multiply": ["$quantity", "$product.price"]}
    }},
    {"$group": {
        "_id": "$product.name",
        "total_revenue": {"$sum": "$revenue"},
        "total_sold": {"$sum": "$quantity"}
    }},
    {"$sort": {"total_revenue": -1}}
]

print("--- Revenue by Product ---")
for doc in orders.aggregate(pipeline):
    print(f"{doc['_id']:<30} | Revenue: ${doc['total_revenue']:>9.2f} | Sold: {doc['total_sold']}")

This pipeline has five stages:
1. **`$lookup`** — joins each order with its product (like SQL JOIN)
2. **`$unwind`** — flattens the joined array into individual documents
3. **`$addFields`** — computes `revenue = quantity × price`
4. **`$group`** — aggregates revenue and quantity by product name
5. **`$sort`** — ranks products by total revenue, highest first

### Step 5 — Monthly Revenue Trend

In [ ]:
# Pipeline: compute revenue per order, then group by month
pipeline = [
    {"$lookup": {
        "from": "products",
        "localField": "product_id",
        "foreignField": "product_id",
        "as": "product"
    }},
    {"$unwind": "$product"},
    {"$addFields": {
        "revenue": {"$multiply": ["$quantity", "$product.price"]},
        "month": {"$substr": ["$date", 0, 7]}
    }},
    {"$group": {
        "_id": "$month",
        "monthly_revenue": {"$sum": "$revenue"},
        "order_count": {"$sum": 1}
    }},
    {"$sort": {"_id": 1}}
]

print("--- Monthly Revenue Trend ---")
for doc in orders.aggregate(pipeline):
    print(f"{doc['_id']}  | Revenue: ${doc['monthly_revenue']:>9.2f} | Orders: {doc['order_count']}")

We use `$substr` to extract the year-month portion (`"2025-01"`) from each order date, then group by that substring. This gives us a month-by-month revenue breakdown without needing date-type parsing.

### Step 6 — Top Customers by Spend

In [ ]:
pipeline = [
    {"$lookup": {
        "from": "products",
        "localField": "product_id",
        "foreignField": "product_id",
        "as": "product"
    }},
    {"$unwind": "$product"},
    {"$addFields": {
        "revenue": {"$multiply": ["$quantity", "$product.price"]}
    }},
    {"$group": {
        "_id": "$customer",
        "total_spend": {"$sum": "$revenue"},
        "orders_count": {"$sum": 1}
    }},
    {"$sort": {"total_spend": -1}},
    {"$limit": 5}
]

print("--- Top 5 Customers by Spend ---")
for i, doc in enumerate(orders.aggregate(pipeline), 1):
    print(f"{i}. {doc['_id']:<10} | Total: ${doc['total_spend']:>9.2f} | Orders: {doc['orders_count']}")

The `$limit` stage restricts output to the top 5 results — useful when you only need the highest-ranked items. Combined with `$sort`, it acts like a "top-N" query.

### Step 7 — Revenue by Region

In [ ]:
pipeline = [
    {"$lookup": {
        "from": "products",
        "localField": "product_id",
        "foreignField": "product_id",
        "as": "product"
    }},
    {"$unwind": "$product"},
    {"$addFields": {
        "revenue": {"$multiply": ["$quantity", "$product.price"]}
    }},
    {"$group": {
        "_id": "$region",
        "total_revenue": {"$sum": "$revenue"},
        "avg_order_value": {"$avg": "$revenue"}
    }},
    {"$sort": {"total_revenue": -1}}
]

print("--- Revenue by Region ---")
for doc in orders.aggregate(pipeline):
    print(f"{doc['_id']:<8} | Revenue: ${doc['total_revenue']:>9.2f} | Avg Order: ${doc['avg_order_value']:.2f}")

Here `$avg` computes the average order value per region alongside `$sum` for total revenue — two aggregations in a single pipeline.

### Step 8 — Category Breakdown with `$match`

In [ ]:
# Use $match early to filter before the join — more efficient
pipeline = [
    {"$lookup": {
        "from": "products",
        "localField": "product_id",
        "foreignField": "product_id",
        "as": "product"
    }},
    {"$unwind": "$product"},
    {"$match": {"product.category": "Electronics"}},
    {"$addFields": {
        "revenue": {"$multiply": ["$quantity", "$product.price"]}
    }},
    {"$group": {
        "_id": "$product.name",
        "total_revenue": {"$sum": "$revenue"},
        "units_sold": {"$sum": "$quantity"}
    }},
    {"$sort": {"total_revenue": -1}}
]

print("--- Electronics Category Breakdown ---")
for doc in orders.aggregate(pipeline):
    print(f"{doc['_id']:<30} | Revenue: ${doc['total_revenue']:>9.2f} | Units: {doc['units_sold']}")

Placing `$match` after `$unwind` lets us filter on the joined product's category. In a production pipeline, you would move `$match` as early as possible (before `$lookup`) for better performance.

### Step 9 — Create Indexes

In [ ]:
# Create an index on product_id in orders for faster joins
orders.create_index("product_id")
print("Created index on orders.product_id")

# Create an index on product_id in products (the foreign key side)
products.create_index("product_id", unique=True)
print("Created unique index on products.product_id")

# Compound index on region + date for filtered time-range queries
orders.create_index([("region", 1), ("date", 1)])
print("Created compound index on (region, date)")

# List all indexes
print("\nOrders indexes:", orders.index_information().keys())
print("Products indexes:", products.index_information().keys())

Indexes are special data structures that speed up query operations at the cost of slightly slower writes. The compound index on `(region, date)` is useful for queries that filter by region and sort by date.

### Step 10 — Analyze with `.explain()`

In [ ]:
# Explain a query that uses our compound index
result = orders.find({"region": "North", "date": {"$gte": "2025-02-01"}}).explain()

print("--- Explain: region + date query ---")
print(f"Execution plan: {result.get('queryPlanner', {}).get('plannerVersion', 'N/A')}")
print(f"Winning plan: {result.get('queryPlanner', {}).get('winningPlan', {}).get('stage', 'N/A')}")

# Explain a query without an index
result2 = orders.find({"customer": "Alice"}).explain()
print(f"\n--- Explain: customer query (no index) ---")
print(f"Winning plan: {result2.get('queryPlanner', {}).get('winningPlan', {}).get('stage', 'N/A')}")

`.explain()` returns the query plan MongoDB would use. A `IXSCAN` stage means an index is being used (good). A `COLLSCAN` means a full collection scan (slower, indicates a missing index).

### Step 11 — Print Summary Report

In [ ]:
# Re-run key pipelines for the summary
revenue_by_product = list(orders.aggregate([
    {"$lookup": {"from": "products", "localField": "product_id", "foreignField": "product_id", "as": "product"}},
    {"$unwind": "$product"},
    {"$addFields": {"revenue": {"$multiply": ["$quantity", "$product.price"]}}},
    {"$group": {"_id": "$product.name", "total_revenue": {"$sum": "$revenue"}}},
    {"$sort": {"total_revenue": -1}}
]))

total_revenue = sum(d["total_revenue"] for d in revenue_by_product)
total_orders = orders.count_documents({})

region_data = list(orders.aggregate([
    {"$lookup": {"from": "products", "localField": "product_id", "foreignField": "product_id", "as": "product"}},
    {"$unwind": "$product"},
    {"$addFields": {"revenue": {"$multiply": ["$quantity", "$product.price"]}}},
    {"$group": {"_id": "$region", "total_revenue": {"$sum": "$revenue"}}},
    {"$sort": {"total_revenue": -1}}
]))

print("         SALES ANALYTICS — SUMMARY REPORT")
print(f"\nTotal orders: {total_orders}")
print(f"Total revenue: ${total_revenue:.2f}")

print("\n--- Top Products by Revenue ---")
for d in revenue_by_product[:5]:
    print(f"  {d['_id']:<30} ${d['total_revenue']:>9.2f}")

print("\n--- Revenue by Region ---")
for d in region_data:
    print(f"  {d['_id']:<10} ${d['total_revenue']:>9.2f}")

print(f"\n--- Indexes Created ---")
print(f"  Orders: {list(orders.index_information().keys())}")
print(f"  Products: {list(products.index_information().keys())}")